In [3]:
import numpy as np
import random
from pyRadPlan.core import CT
from pyRadPlan.stf import SteeringInformation, Beam, Ray
from pyRadPlan.machines import MachineExternalIon
from pyRadPlan.dose.engines import DoseEngineHongPencilBeam
from pyRadPlan import IonPlan

# ==========================================
# 1. CREATE A CUSTOM 3D CT BLOCK (e.g., Water Tank)
# ==========================================
# Dimensions: 24 x 24 x 150 voxels
voxels_shape = (24, 24, 150) 
voxel_size_mm = [2.0, 2.0, 2.0] # 2mm cubic voxels

# Create a uniform density grid (e.g., 0 HU = Water)
# You can randomize this array to simulate tissue layers (bone, lung, etc.)
raw_hu_data = np.zeros(voxels_shape, dtype=np.float32)

# Instantiate pyRadPlan's internal CT object
ct = CT(
    image=raw_hu_data,
    spacing=voxel_size_mm,
    origin=[0.0, 0.0, 0.0],
    direction=np.eye(3).tolist()
)

# ==========================================
# 2. CONSTRUCT A PLAN & LOAD BEAM MACHINE
# ==========================================
plan = IonPlan(radiation_mode="protons", machine="Generic")
# Load the standard proton machine data from the toolkit
machine = MachineExternalIon.load_from_file("Generic_Protons.json") 

# ==========================================
# 3. DEFINE A RANDOM INDIVIDUAL PENCIL BEAM
# ==========================================
# Randomize energy between 70 MeV and 220 MeV (typical proton therapy ranges)
random_energy = random.uniform(70.0, 220.0)

# Create a single beam at a specific angle (e.g., Gantry 0, Couch 0)
beam = Beam(
    gantry_angle=0.0,
    couch_angle=0.0,
    bixel_width=5.0,
    radiation_mode="protons"
)

# Create a single Ray (pencil beam) pointing down the center of our block
# ray_pos dictates the entry coordinates [x, y, z]
center_x = (voxels_shape[0] * voxel_size_mm[0]) / 2.0
center_y = (voxels_shape[1] * voxel_size_mm[1]) / 2.0
entry_z = 0.0

ray = Ray(
    ray_pos=[center_x, center_y, entry_z],
    energy=random_energy,
    weight=1.0 # Set a baseline weight/intensity for forward calculation
)
beam.rays = [ray]

# Package it into the Steering Information structure
stf = SteeringInformation(beams=[beam])

# ==========================================
# 4. RUN THE LOW-LEVEL FORWARD DOSE ENGINE
# ==========================================
# Instantiate the standard pencil beam engine
engine = DoseEngineHongPencilBeam()

print(f"Calculating dose for a single beam with Energy: {random_energy:.2f} MeV...")

# Calculate the actual physical 3D dose grid distribution
# This completely bypasses the massive Dij matrix loop
dose_result = engine.calc_dose_forward(ct, stf, plan, machine)

# ==========================================
# 5. EXTRACT YOUR RANGE DATA
# ==========================================
# dose_result.cube contains the exact 3D numpy array of the dose distribution
dose_matrix = dose_result.cube

print("Calculation finished successfully!")
print(f"Output dose matrix shape: {dose_matrix.shape}")


ModuleNotFoundError: No module named 'pyRadPlan'